In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms

import numpy as np
import pandas as pd

import random

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

device(type='mps')

## Data Load + Transform 

In [3]:
BATCH_SIZE = 128


transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize(0.5, 0.5, 0.5)
])

try:
    dataset = ImageFolder(root="data/", transform=transform)
    print(f"Loaded train dataset: {len(dataset)} images, {len(dataset.classes)} classes.")
except FileNotFoundError:
    print("Couldnt find 'data/' directory!!!!")

Loaded train dataset: 39209 images, 43 classes.


In [5]:
train_dataset, test_dataset = random_split(dataset=dataset, lengths=[0.9, 0.1])
train_dataset, val_dataset = random_split(dataset=train_dataset, lengths=[8/9, 1/9])

In [11]:
for x, y in train_dataset:
    print(f"Input img shape: {x.shape}")
    print(f"class: {y}")
    break

Input img shape: torch.Size([3, 32, 32])
class: 31


In [12]:
data_loaders = {
    "train": DataLoader(dataset=train_dataset, shuffle=True, batch_size=BATCH_SIZE, pin_memory=True, num_workers=4),
    "val": DataLoader(dataset=val_dataset, shuffle=False, batch_size=BATCH_SIZE, num_workers=4),
    "test": DataLoader(dataset=test_dataset, shuffle=False, batch_size=BATCH_SIZE, num_workers=4)
}

## Architecture

### AutoEncoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, input_channels, hidden_channels, latent_dim):
        super().__init__()
        
        self.input_size = input_size
        self.input_channels = input_channels
        self.hidden_channels = hidden_channels
        self.latent_dim = latent_dim
        
        self.encode = nn.Sequential([
            nn.Conv2d(in_channels=input_channels, out_channels=hidden_channels, kernel_size=(3,3), stride=2, padding=1), # 32x32 -> 16x16
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=hidden_channels, out_channels=hidden_channels, kernel_size=(2,2), padding=1), # 16x16 -> 9x9
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=hidden_channels, out_channels=2*hidden_channels, kernel_size=(2,2), padding=1), # 9x9 -> 10x10
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=2*hidden_channels, out_channels=2*hidden_channels, kernel_size=(2,2), stride=2), # 10x10 -> 5x5
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(in_features=2*hidden_channels*5*5, out_features=latent_dim)
        ])

    def forward(self, x):
        return self.encode(x)


In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_size, input_channels, hidden_channels, latent_dim):
        super().__init__()
        
        self.input_size = input_size
        self.input_channels = input_channels
        self.hidden_channels = hidden_channels
        self.latent_dim = latent_dim
        
        self.encode = nn.Sequential([
            nn.Conv2d(in_channels=input_channels, out_channels=hidden_channels, kernel_size=(3,3), stride=2, padding=1), # 32x32 -> 16x16
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=hidden_channels, out_channels=hidden_channels, kernel_size=(2,2), padding=1), # 16x16 -> 9x9
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=hidden_channels, out_channels=2*hidden_channels, kernel_size=(2,2), padding=1), # 9x9 -> 10x10
            nn.LeakyReLU(),
            nn.Conv2d(in_channels=2*hidden_channels, out_channels=2*hidden_channels, kernel_size=(2,2), stride=2), # 10x10 -> 5x5
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(in_features=2*hidden_channels*5*5, out_features=latent_dim)
        ])


### VariationalAutoEncoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, input_channels, hidden_dim, latent_dim):
        super().__init__()
        
        self.input_dim = input_size * input_size * input_channels
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        self.fc1 = nn.Linear(in_features=self.input_dim, out_features=self.hidden_dim*2)
        self.fc2 = nn.Linear(in_features=self.hidden_dim*2, out_features=self.hidden_dim)
        self.fc3 = nn.Linear(in_features=self.hidden_dim, out_features=self.hidden_dim)
        self.fc_mean = nn.Linear(in_features=self.hidden_dim, out_features=self.latent_dim)
        self.fc_logvar = nn.Linear(in_features=self.hidden_dim, out_features=self.latent_dim)

        self.act = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        mean = self.fc_mean(x)
        log_var = self.fc_logvar(x)
        
        return mean, log_var


In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_size, output_channels):
        super().__init__()
        
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.output_size = output_size
        self.output_channels = output_channels
        self.output_dim = output_size * output_size * output_channels
        
        self.fc1 = nn.Linear(in_features=self.latent_dim, out_features=self.hidden_dim)
        self.fc2 = nn.Linear(in_features=self.hidden_dim, out_features=self.hidden_dim)
        self.fc3 = nn.Linear(in_features=self.hidden_dim, out_features=self.hidden_dim*2)
        self.fc4 = nn.Linear(in_features=self.hidden_dim*2, out_features=self.output_dim)

        self.act = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        x_hat = torch.sigmoid(self.fc4(x))
        x_hat = x_hat.view([-1, self.output_channels, self.output_size, self.output_size])
        return x_hat

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_size, input_channels, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder(input_size=input_size, input_channels=input_channels, hidden_dim=hidden_dim, latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim, hidden_dim=hidden_dim, output_size=input_size, output_channels=input_channels)

    def reparametrize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)   # exp( log var ) --> var, sqrt(var) --> std
        eps = torch.rand_like(std)
        return mean + std * eps
    
    def forward(self, x):
        mean, logvar = self.encoder(x)
        z = self.reparametrize(mean=mean, logvar=logvar)
        x_hat = self.decoder(z)
        return x_hat

Normal distribution: 

For z∼N(μ,σ2), we can use:

z = μ+σϵ, ϵ∼N(0,1)

In [ ]:
""" My intuition behind reparametrization """

# given the assumption that latent space params are all from a given disribution, meaning last linear projection of encoder produces EX/mean and variance 
# - we cannot properly train decoder only on those values -> at beginning, params/weights of linear layer would be totally random, wouldn't fit in e.g. normal distribution 
# so passing them to decoder and training would reasult in having classic Autoencoder, without ability to sample latent vector
# we also cannot pass to decoder random sample from e.g. normal distribution, that way it wouldn't be connected with input data that should be reconstructed,
# that's why we sample normal disribution and scale it by 'mean' and 'variance' produced from last linear projection so we produce decoder input as scaled/shifted normal distro
# (scaled/shifted to input distro) - decoder learns to reconstruct orginal data from that distro, so after all to minimize reconstruction error, 
# params are adjusted and in next iters last linear projection producing mean and var gets better and better, finally producing proper gaussian distribution

"""
Reparametrization:

What about the sampling of the latent vector? 
Here’s there is a fundamental problem to the implementation. We are going to use back-propagation to tune the parameters ϕ and θ 
but sampling from latent space is an stochastic process that depends on ϕ! 
Instead of using complicating methods, we will tackle this problem with the reparametrization trick. 
The key idea is that we can move all the randomness to a distribution that does not change with the parameters, drawing instead a random variable

Then, to reproduce the sampling from

we have to shift the sample from ε and scale its variance

which holds because

You may have noticed that the layer that produces the variance is log_var . This is because we can write

There are various reasons to use the log variance instead of directly the variance, for example:
The variance is always positive, so we would have to force the layer in the encoder to only output positive values. Instead, the logarithm can be negative. 
Furthermore, taking the exponential in the last equation always leads to a positive number
We avoid problems with the square root of a negative number.
"""